# Trabajo Práctico Final — Introducción a la Ingeniería de Datos

**Autora:** Bioing. Aylin Agatha Vazquez Chenlo — Doctorado en Ingeniería, ITBA

**Profesor:** Dr. Nicolás Guillermo Tripp

**Repositorio:** https://github.com/aylinavch/ITBA-Introduccion-Ingenieria-de-Datos

Cada estudiante deberá entregar, con al menos 1 semana de anticipación a la defensa, un enlace a un cuaderno de Google Colab público dentro del repositorio en GitHub declarado en el cursado, debidamente documentado mediante celdas de texto Markdown y estrictamente reproducible.

El cuaderno debe cumplir obligatoriamente con los siguientes requisitos:

## **Declaración de Uso de Inteligencia Artificial (IA)**

El uso de IA generativa está permitido y fomentado, siempre que se declare de forma transparente. El cuaderno debe incluir una celda de texto al inicio titulada "Declaración de uso de IA", listando cada herramienta utilizada y su propósito específico.

- Formato exigido: "[Nombre y versión de la IA]" para "[Propósito exacto]".
- Ejemplos: "Claude Opus 4.6" para "mejorar redacción de textos de las justificaciones", "Chat GPT 5.6" para "crear código fuente para expresiones regulares", o "GitHub Copilot" para "autocompletar código de gráficos en Plotly".

**Instrucciones de la IA**: Debe presentar una guía de instrucciones que permita reproducir lo mas fiel posible la información generada mediante inteligencia artificial. Esta guía debe demostrar la capacidad del estudiante para dirigir a la herramienta en al creación del resultado obtenido (Human-in-the-loop). Se espera que las instrucciones incluyan el vocabulario especifico del curso.
No se aceptan instrucciones que sean una copia directa de los enunciados de actividades.

## **Datos**

Debe utilizar un dataset crudo disponible desde un repositorio en la nube o fuente pública (GitHub, Zenodo, OSF, API, etc.). El dataset debe ser de un tema afín a su trabajo de tesis. No se aceptarán cuadernos que requieran subir archivos manualmente o que dependan de rutas locales/privadas (ej. carpetas privadas de Google Drive).

## **Extracción, Transformación y Carga (ETL)**

Se debe realizar el proceso completo de ETL, incluyendo:

- Carga de datos optimizando los parámetros (elección de columnas, tipado de variables).
- Identificación y resolución de valores faltantes (NAs/NULLs) y datos atípicos (outliers). Cada decisión de imputación, transformación o descarte debe estar justificada metodológicamente en celdas de texto.
- Almacenamiento de los datos en una base datos transaccional SQLite normalizada como mínimo en 2NF.
- Creación y carga de los datos en un almacén de datos SQLite, siguiendo los principios del modelado dimensional (esquema estrella o copo de nieve), separando claramente las tablas de hechos de las tablas de dimensiones.

## **Visualización**

Utilizando los datos del almacén deberá construir una interfaz interactiva de visualización (streamlit, posix, shiny, dash, postly) que transmita un mensaje claro sobre su investigación.

## **Límite de páginas** 
15

---

**Utilización de IA para co-desarrollo:** Se utilizó **Claude Opus 5.5** (Anthropic) para:
- Corregir la redacción de las explicaciones/justificaciones en las celdas markdown del notebook.

---

## Contexto de investigación

**Dataset elegido:** *The Bitbrain Open Access Sleep (BOAS) dataset* (López-Larraz et al.), OpenNeuro `ds005555` v1.1.3, DOI [10.18112/openneuro.ds005555.v1.1.3](https://doi.org/10.18112/openneuro.ds005555.v1.1.3), licencia CC0.
- 128 noches de 100 personas adultas sanas, registradas en simultáneo con una PSG clínica (Micromed) y una vincha de EEG wearable (Bitbrain).
- Cada época de 30 s tiene tres etiquetas de estadio: el consenso de tres expertos sobre la PSG (criterios AASM), la estadificación automática con IA sobre la PSG y la estadificación automática con IA sobre la vincha.
- Los datos están en formato [BIDS](https://bids-specification.readthedocs.io/en/stable/): `participants.tsv` (edad, sexo, IMC) y un `events.tsv` por noche y dispositivo.

**Pregunta:** ¿Qué tan bien estadifica el sueño un dispositivo wearable de EEG frente al estándar clínico (polisomnografía puntuada por expertos), y cómo varía esa concordancia según el estadio de sueño y las características de la persona (edad, sexo, IMC)?

**Relación con mi tesis:** Mi tesis aborda el procesamiento de señales de sueño, principalmente EEG. Validar dispositivos portátiles contra la polisomnografía (PSG) es un paso necesario para llevar el monitoreo del sueño fuera del laboratorio.

---

## Set Up

Instalación de las dependencias que no vienen preinstaladas en Google Colab y carga de librerías. La lista completa de dependencias está en [`TP-Final/requirements.txt`](https://github.com/aylinavch/ITBA-Introduccion-Ingenieria-de-Datos/blob/main/TP-Final/requirements.txt): si el archivo está disponible localmente (ejecución desde el repositorio clonado) se usa esa copia; si no (Colab), se descarga desde GitHub.

In [3]:
# Instalación de dependencias
import os, urllib.request

REQ_URL = "https://raw.githubusercontent.com/aylinavch/ITBA-Introduccion-Ingenieria-de-Datos/main/TP-Final/requirements.txt"
REQ_PATH = "requirements.txt"

if not os.path.exists(REQ_PATH):
    urllib.request.urlretrieve(REQ_URL, REQ_PATH)  # En Colab el repositorio no está clonado

%pip install -q -r {REQ_PATH}


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
# Importación de librerías
# Librería estándar
import os
import sqlite3
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

# Ingesta y manipulación
import numpy as np
import pandas as pd
import requests

# Análisis
from sklearn.metrics import cohen_kappa_score, confusion_matrix
from sklearn.ensemble import IsolationForest

# Visualización interactiva
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output

# Configuración general
SEED = 42                                   # Semilla para reproducibilidad
np.random.seed(SEED)
pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 2)   # float32 se muestra sin ruido de redondeo
EN_COLAB = "google.colab" in str(get_ipython())

import sklearn, plotly, dash
print(f"Entorno: {'Google Colab' if EN_COLAB else 'local'}")
for lib in (pd, np, sklearn, plotly, dash):
    print(f"  {lib.__name__:<8} {lib.__version__}")

Entorno: local
  pandas   3.0.5
  numpy    2.5.3
  sklearn  1.9.1
  plotly   7.1.0
  dash     4.5.0


---

## 1. Extracción

**Fuente.** Los archivos se descargan desde la réplica oficial de OpenNeuro en GitHub ([`OpenNeuroDatasets/ds005555`](https://github.com/OpenNeuroDatasets/ds005555)), fijando el *tag* `1.1.3` en lugar de la rama `main`: cualquier ejecución futura lee exactamente la misma versión publicada, aunque el dataset se actualice.

**Alcance.** Se descargan `participants.tsv` (una fila por noche registrada) y los 256 `events.tsv` (128 noches × 2 dispositivos), que contienen la estadificación por época de 30 s. Las señales crudas (`.edf`, ~180 MB por noche) **no** se descargan porque la pregunta de investigación se responde con las etiquetas de estadio.

**Caché.** Cada archivo se guarda en `data/raw/ds005555/`; si ya existe, no se vuelve a descargar.

In [5]:
# Parámetros de la fuente y estructura de carpetas
DATASET = "ds005555"
VERSION = "1.1.3"   # tag de la versión publicada en OpenNeuro
BASE_URL = f"https://raw.githubusercontent.com/OpenNeuroDatasets/{DATASET}/{VERSION}"

DIR_RAW = Path("data/raw") / DATASET
DIR_PROCESSED = Path("data/processed")
for carpeta in (DIR_RAW, DIR_PROCESSED):
    carpeta.mkdir(parents=True, exist_ok=True)

def descargar(ruta_relativa: str) -> Path:
    """Descarga un archivo del dataset si no está en caché y devuelve su ruta local."""
    destino = DIR_RAW / ruta_relativa
    if not destino.exists():
        destino.parent.mkdir(parents=True, exist_ok=True)
        respuesta = requests.get(f"{BASE_URL}/{ruta_relativa}", timeout=30)
        respuesta.raise_for_status()  # Falla explícitamente ante un error HTTP
        destino.write_bytes(respuesta.content)
    return destino

def memoria_mb(df: pd.DataFrame) -> float:
    """Memoria RAM ocupada por un DataFrame (incluye el contenido de los strings)."""
    return df.memory_usage(deep=True).sum() / 1024**2

### 1.1 Participantes (`participants.tsv`)

Por convención BIDS, `participant_id` identifica a cada **noche registrada** (`sub-1` … `sub-128`), mientras que `pid` identifica a la **persona** (algunas personas durmieron más de una noche). Los parámetros de carga se eligen a partir del diccionario de datos (`participants.json`):

| Columna | Tipo | Justificación |
|---|---|---|
| `participant_id` | `string` | Identificador textual de la sesión. |
| `pid` | `Int16` | Entero nullable; 100 personas. |
| `age` | `Int8` | Edad en años (18–82) cabe en 8 bits; *nullable* para admitir faltantes. |
| `sex` | `category` | Solo dos niveles (`F`, `M`). |
| `bmi` | `float32` | Un decimal de precisión; `float64` es innecesario. |

`na_values=["n/a"]` interpreta explícitamente el marcador de faltante que usa BIDS.

In [7]:
descargar("participants.json")  # Diccionario de variables (metadatos)

participantes = pd.read_csv(
    descargar("participants.tsv"),
    sep="\t",
    usecols=["participant_id", "pid", "age", "sex", "bmi"],
    dtype={"participant_id": "string", "pid": "Int16", "age": "Int8",
           "sex": "category", "bmi": "float32"},
    na_values=["n/a"],
)
print(f"{len(participantes)} noches registradas | {participantes['pid'].nunique()} personas | "
      f"{memoria_mb(participantes):.3f} MB")
participantes.head()

128 noches registradas | 100 personas | 0.003 MB


,participant_id,age,sex,bmi,pid
0,sub-1,27,F,19.799999,89
1,sub-2,27,F,19.799999,89
2,sub-3,27,M,25.100000,1
3,sub-4,26,F,22.000000,2
4,sub-5,27,F,20.700001,3


### 1.2 Hipnogramas (`events.tsv`)

Cada noche tiene un archivo por dispositivo: `acq-psg` (polisomnografía) y `acq-headband` (vincha wearable). Ambos tienen una fila por época de 30 s con las columnas `onset`, `duration`, `begsample`, `endsample`, `offset` y las etiquetas de estadio (`stage_hum`, solo en PSG; `stage_ai`). Las descargas se hacen en paralelo porque son operaciones de entrada/salida.

In [10]:
SESIONES = participantes["participant_id"].tolist()
ADQUISICIONES = ["psg", "headband"]

def ruta_eventos(sesion: str, acq: str) -> str:
    return f"{sesion}/eeg/{sesion}_task-Sleep_acq-{acq}_events.tsv"

with ThreadPoolExecutor(max_workers=8) as pool:
    archivos = list(pool.map(descargar, [ruta_eventos(s, a) for s in SESIONES for a in ADQUISICIONES]))
print(f"{len(archivos)} archivos de eventos disponibles en {DIR_RAW}/")

256 archivos de eventos disponibles en data/raw/ds005555/


**Carga ingenua vs. optimizada.** Primero se cargan los archivos con la configuración por defecto de pandas para verificar que las columnas que se van a descartar son redundantes:

- `duration` vale siempre 30 s y `offset` siempre 0 (constantes, sin información).
- `begsample` y `endsample` se deducen de `onset` y de la frecuencia de muestreo (256 Hz, declarada en los `*_eeg.json`).

Luego se cargan solo `onset` y las etiquetas, con enteros de 8 y 32 bits.

In [11]:
FS = 256  # Hz, frecuencia de muestreo declarada en *_eeg.json

def leer_eventos(sesion: str, acq: str, optimizado: bool = True) -> pd.DataFrame:
    ruta = DIR_RAW / ruta_eventos(sesion, acq)
    if not optimizado:
        return pd.read_csv(ruta, sep="\t")
    columnas = ["onset", "stage_hum", "stage_ai"] if acq == "psg" else ["onset", "stage_ai"]
    return pd.read_csv(ruta, sep="\t", usecols=columnas, na_values=["n/a"],
                       dtype={"onset": "int32", "stage_hum": "Int8", "stage_ai": "Int8"})

# Carga ingenua: todas las columnas con tipos por defecto (int64)
ingenuo = pd.concat({(s, a): leer_eventos(s, a, optimizado=False)
                     for s in SESIONES for a in ADQUISICIONES},
                    names=["participant_id", "acq", "fila"])

# Validación de las columnas que se descartan
assert (ingenuo["duration"] == 30).all()
assert (ingenuo["offset"] == 0).all()
assert (ingenuo["begsample"] == ingenuo["onset"] * FS + 1).all()
assert (ingenuo["endsample"] == (ingenuo["onset"] + 30) * FS).all()
print("✓ duration, offset, begsample y endsample son redundantes con onset")

✓ duration, offset, begsample y endsample son redundantes con onset


In [12]:
# Carga optimizada: proyección de columnas + tipos reducidos
def cargar_dispositivo(acq: str) -> pd.DataFrame:
    df = pd.concat({s: leer_eventos(s, acq) for s in SESIONES}, names=["participant_id", "fila"])
    return df.reset_index(level="participant_id").reset_index(drop=True)

psg = cargar_dispositivo("psg").rename(columns={"stage_ai": "stage_ai_psg"})
vincha = cargar_dispositivo("headband").rename(columns={"stage_ai": "stage_ai_hb"})

# Unión época a época: cada época debe existir en ambos dispositivos (relación 1:1)
eventos = psg.merge(vincha, on=["participant_id", "onset"], how="outer",
                    validate="one_to_one", indicator=True)
assert (eventos["_merge"] == "both").all(), "Hay épocas presentes en un solo dispositivo"
eventos = eventos.drop(columns="_merge")
eventos["participant_id"] = eventos["participant_id"].astype("category")

print(f"Carga ingenua   : {memoria_mb(ingenuo):6.2f} MB ({len(ingenuo):,} filas, PSG y vincha por separado)")
print(f"Carga optimizada: {memoria_mb(eventos):6.2f} MB ({len(eventos):,} épocas, PSG y vincha unidos)")
del ingenuo

Carga ingenua   :  13.98 MB (240,190 filas, PSG y vincha por separado)
Carga optimizada:   1.38 MB (120,095 épocas, PSG y vincha unidos)


### 1.3 Tabla única de extracción y controles de integridad

Se unen las épocas con los atributos de cada noche en una única tabla plana (*One Big Table*), que es el punto de partida de la Transformación. Antes de avanzar se verifican las reglas de integridad que el resto del pipeline asume.

In [14]:
obt = eventos.merge(participantes.astype({"participant_id": "category"}),
                    on="participant_id", how="left", validate="many_to_one")

# Controles de integridad
assert obt["participant_id"].nunique() == 128 and obt["pid"].nunique() == 100
assert obt.groupby("participant_id", observed=True)["onset"].diff().dropna().eq(30).all(), \
    "Hay saltos entre épocas consecutivas"
assert set(obt["stage_hum"].unique()) <= {0, 1, 2, 3, 4, 8}
assert set(obt["stage_ai_psg"].unique()) | set(obt["stage_ai_hb"].unique()) <= {-2, 0, 1, 2, 3, 4}
print("✓ 128 noches, 100 personas, épocas contiguas y códigos de estadio dentro del dominio")

obt.to_parquet(DIR_PROCESSED / "01_extraccion_obt.parquet", index=False)
print(f"OBT: {obt.shape[0]:,} filas × {obt.shape[1]} columnas | {memoria_mb(obt):.2f} MB")
obt.head()

✓ 128 noches, 100 personas, épocas contiguas y códigos de estadio dentro del dominio
OBT: 120,095 filas × 9 columnas | 2.52 MB


,participant_id,onset,stage_hum,stage_ai_psg,stage_ai_hb,age,sex,bmi,pid
0,sub-1,0,3,3,3,27,F,19.8,89
1,sub-1,30,3,3,3,27,F,19.8,89
2,sub-1,60,3,3,3,27,F,19.8,89
3,sub-1,90,3,3,3,27,F,19.8,89
4,sub-1,120,3,3,3,27,F,19.8,89


### 1.4 Primer diagnóstico de calidad

En esta etapa solo se **identifican** los problemas; su resolución se justifica en la sección de Transformación.

In [15]:
CODIGOS = {"stage_hum": 8, "stage_ai_psg": -2, "stage_ai_hb": -2}  # 8 = desconexión del PSG, -2 = artefacto
print("Faltantes en participants.tsv:")
print(participantes[participantes.isna().any(axis=1)].to_string(index=False))

print("\nÉpocas con código especial (no son un estadio de sueño):")
for col, cod in CODIGOS.items():
    n = (obt[col] == cod).sum()
    print(f"  {col:<13} = {cod:>2}: {n:>5,} épocas ({n / len(obt):.1%})")

horas = obt.groupby("participant_id", observed=True).size() * 30 / 3600
print(f"\nDuración del registro por noche (h): mediana {horas.median():.1f} | "
      f"mín {horas.min():.1f} ({horas.idxmin()}) | máx {horas.max():.1f}")

primer_estadio = obt.groupby("participant_id", observed=True)["stage_hum"].first()
print(f"Noches que no comienzan en vigilia: {primer_estadio[primer_estadio != 0].to_dict()}")

por_persona = participantes.groupby("pid")[["age", "bmi"]].nunique()
print("\nPersonas con más de un valor de edad o IMC entre noches:")
print(participantes[participantes["pid"].isin(por_persona[(por_persona > 1).any(axis=1)].index)]
      .to_string(index=False))

Faltantes en participants.tsv:
participant_id  age sex  bmi  pid
        sub-16 <NA>   F 26.8   11
        sub-29   23   F  NaN   21

Épocas con código especial (no son un estadio de sueño):
  stage_hum     =  8:   336 épocas (0.3%)
  stage_ai_psg  = -2:   550 épocas (0.5%)
  stage_ai_hb   = -2: 4,881 épocas (4.1%)

Duración del registro por noche (h): mediana 7.8 | mín 3.4 (sub-8) | máx 9.3
Noches que no comienzan en vigilia: {'sub-1': 3}

Personas con más de un valor de edad o IMC entre noches:
participant_id  age sex  bmi  pid
        sub-23   34   M 28.1   15
       sub-124   35   M 27.3   15


**Hallazgos** (se resuelven en la Transformación):

1. **Faltantes:** una noche sin edad (`sub-16`) y una sin IMC (`sub-29`).
2. **Códigos especiales:** las etiquetas `8` (desconexión del PSG, p. ej. para ir al baño) y `-2` (artefacto o dato faltante en la IA) no son estadios de sueño. Son NAs codificados como números y se tratan como faltantes. La vincha concentra la mayor proporción de artefactos.
3. **Atípico de duración:** `sub-8` registra 3,4 h, muy por debajo de la mediana (~7,8 h).
4. **Inicio de registro:** `sub-1` comienza directamente en N3 (las otras 127 noches comienzan en vigilia), lo que indica que el registro empezó con la persona ya dormida; su latencia de sueño no es calculable.
5. **Dependencias:** el sexo es constante por persona, pero la edad y el IMC pueden cambiar entre noches de la misma persona (`pid` 15). Esto guía la normalización: el sexo es atributo de la *persona*; la edad y el IMC, de la *noche*.

---